# HW04 深度学习作业 4

本 notebook 按 HW04.pdf 的题目顺序完成：序列模型、循环神经网络、高级循环神经网络、嵌入向量和注意力机制。理论题给出推导过程，编程题给出可直接运行的实现与中文输出。


In [1]:
import math
import re
from collections import Counter, defaultdict

import numpy as np
import torch
from torch import nn

np.set_printoptions(precision=4, suppress=True)
torch.set_printoptions(precision=4, sci_mode=False)
torch.manual_seed(42)

print("环境准备完成：NumPy 和 PyTorch 已导入。")


环境准备完成：NumPy 和 PyTorch 已导入。


## 2 序列模型

### 2.1 理论计算题

给定字符序列 ababc，采用一阶马尔可夫模型，只统计相邻字符转移：

$$
a \to b,\quad b \to a,\quad a \to b,\quad b \to c.
$$

词汇表为 $\{a,b,c\}$，大小 $|V|=3$。当前一个字符为 b 时，观测到

$$
N(b\to a)=1,\quad N(b\to b)=0,\quad N(b\to c)=1,\quad N(b)=2.
$$

使用拉普拉斯平滑后，

$$
p(x\mid b)=\frac{N(b\to x)+1}{N(b)+|V|}.
$$

所以

$$
p(a\mid b)=\frac{1+1}{2+3}=0.4,
\qquad
p(c\mid b)=\frac{1+1}{2+3}=0.4.
$$

未出现的转移 b 到 b 也被考虑，其平滑概率为 $\frac{1}{5}=0.2$。


In [2]:
sequence = "ababc"
vocabulary = sorted(set(sequence))

# 统计所有相邻字符的转移次数。
transition_counts = defaultdict(Counter)
for previous_char, current_char in zip(sequence[:-1], sequence[1:]):
    transition_counts[previous_char][current_char] += 1


def laplace_transition_probability(previous_char, current_char, vocabulary):
    """计算一阶马尔可夫转移概率，并使用拉普拉斯平滑。"""
    total_count = sum(transition_counts[previous_char].values())
    vocabulary_size = len(vocabulary)
    transition_count = transition_counts[previous_char][current_char]
    return (transition_count + 1) / (total_count + vocabulary_size)

print("序列：", sequence)
print("词汇表：", vocabulary)
print("从字符 b 出发的原始转移次数：", dict(transition_counts["b"]))
print("加一平滑后 p('a' | 'b') =", laplace_transition_probability("b", "a", vocabulary))
print("加一平滑后 p('c' | 'b') =", laplace_transition_probability("b", "c", vocabulary))
print("加一平滑后 p('b' | 'b') =", laplace_transition_probability("b", "b", vocabulary))


序列： ababc
词汇表： ['a', 'b', 'c']
从字符 b 出发的原始转移次数： {'a': 1, 'c': 1}
加一平滑后 p('a' | 'b') = 0.4
加一平滑后 p('c' | 'b') = 0.4
加一平滑后 p('b' | 'b') = 0.2


### 2.2 编程题：文本预处理与自回归样本构造

函数 preprocess_text(text, n) 完成四步：转小写并去除标点符号、按空格分词、按出现频率构建词汇表、用长度为 n 的滑动窗口生成特征和下一个词标签。最后一个窗口没有后续词时，标签记为 None。


In [3]:
def preprocess_text(text, n):
    """预处理英文文本，并生成自回归语言模型样本。"""
    if n <= 0:
        raise ValueError("窗口长度 n 必须是正整数。")

    # 只保留英文字母和空格；标点会被替换为空格，避免单词粘连。
    cleaned_text = re.sub(r"[^a-zA-Z\s]+", " ", text.lower())
    tokens = cleaned_text.split()

    # 构建按频率排序的词汇表；同频词按首次出现位置排序，保证结果稳定。
    frequencies = Counter(tokens)
    first_position = {}
    for index, token in enumerate(tokens):
        first_position.setdefault(token, index)
    sorted_words = sorted(frequencies, key=lambda word: (-frequencies[word], first_position[word]))
    vocabulary = {word: index for index, word in enumerate(sorted_words)}

    # 生成长度为 n 的特征窗口，并取窗口后的第一个词作为标签。
    features = []
    labels = []
    for start in range(0, max(len(tokens) - n + 1, 0)):
        feature_window = tokens[start:start + n]
        next_position = start + n
        label = tokens[next_position] if next_position < len(tokens) else None
        features.append(feature_window)
        labels.append(label)

    return vocabulary, features, labels

example_text = "The time machine"
example_vocabulary, example_features, example_labels = preprocess_text(example_text, n=2)

print("示例文本：", example_text)
print("词汇表：", example_vocabulary)
print("特征列表：", example_features)
print("标签列表：", example_labels)

longer_text = "The Time Machine, by H. G. Wells! The time traveller begins his story."
vocabulary, features, labels = preprocess_text(longer_text, n=3)

print("\n扩展示例文本：", longer_text)
print("扩展示例词汇表：", vocabulary)
print("前三个特征窗口：", features[:3])
print("前三个标签：", labels[:3])


示例文本： The time machine
词汇表： {'the': 0, 'time': 1, 'machine': 2}
特征列表： [['the', 'time'], ['time', 'machine']]
标签列表： ['machine', None]

扩展示例文本： The Time Machine, by H. G. Wells! The time traveller begins his story.
扩展示例词汇表： {'the': 0, 'time': 1, 'machine': 2, 'by': 3, 'h': 4, 'g': 5, 'wells': 6, 'traveller': 7, 'begins': 8, 'his': 9, 'story': 10}
前三个特征窗口： [['the', 'time', 'machine'], ['time', 'machine', 'by'], ['machine', 'by', 'h']]
前三个标签： ['by', 'h', 'g']


## 3 循环神经网络

### 3.1 理论计算题

线性 RNN 定义为

$$
h_t=W_{hh}h_{t-1}+W_{hx}x_t,
\qquad
 o_t=W_{oh}h_t.
$$

平方损失为

$$
L=\frac{1}{2}\sum_{t=1}^{T}(o_t-y_t)^2.
$$

令输出误差为

$$
e_t=\frac{\partial L}{\partial o_t}=o_t-y_t.
$$

定义总隐藏梯度

$$
\delta_t=\frac{\partial L}{\partial h_t}.
$$

时间反向传播给出递推式

$$
\delta_t=W_{oh}^{\top}e_t+W_{hh}^{\top}\delta_{t+1},
\qquad
\delta_{T+1}=0.
$$

由于 $W_{hh}$ 在所有时间步共享，总梯度为

$$
\frac{\partial L}{\partial W_{hh}}=\sum_{t=1}^{T}\delta_t h_{t-1}^{\top}.
$$

进一步展开得到

$$
\delta_t=\sum_{k=t}^{T}(W_{hh}^{\top})^{k-t}W_{oh}^{\top}(o_k-y_k),
$$

因此

$$
\boxed{
\frac{\partial L}{\partial W_{hh}}
=\sum_{t=1}^{T}
\left[
\sum_{k=t}^{T}(W_{hh}^{\top})^{k-t}W_{oh}^{\top}(o_k-y_k)
\right]h_{t-1}^{\top}
}
$$

梯度消失或爆炸主要由反复相乘的 $(W_{hh}^{\top})^{k-t}$ 决定。若 $W_{hh}$ 的谱半径或矩阵范数明显小于 1，长距离梯度会消失；若明显大于 1，长距离梯度会爆炸；若尺度接近 1，梯度相对更稳定。


### 3.2 编程题：简单 RNN 单元的前向传播与反向传播

实现一个 tanh RNN 单元：

$$
z_t=x_tW_{hx}+h_{t-1}W_{hh}+b_h,
\qquad
h_t=\tanh(z_t).
$$

其中 x_t 的形状为 (batch_size, input_size)，h_prev 的形状为 (batch_size, hidden_size)，W_hx 的形状为 (input_size, hidden_size)，W_hh 的形状为 (hidden_size, hidden_size)。


In [4]:
def rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h):
    """计算一个 tanh RNN 单元的前向传播。"""
    z_t = x_t @ W_hx + h_prev @ W_hh + b_h
    h_t = np.tanh(z_t)
    cache = {
        "x_t": x_t,
        "h_prev": h_prev,
        "W_hx": W_hx,
        "W_hh": W_hh,
        "h_t": h_t,
    }
    return h_t, cache


def rnn_cell_backward(dh_next, cache):
    """根据上游梯度计算 RNN 单元中各变量的梯度。"""
    x_t = cache["x_t"]
    h_prev = cache["h_prev"]
    W_hx = cache["W_hx"]
    W_hh = cache["W_hh"]
    h_t = cache["h_t"]

    # tanh 的导数为 1 - tanh(z)^2，这里 h_t 已经等于 tanh(z_t)。
    dz_t = dh_next * (1 - h_t ** 2)

    dx_t = dz_t @ W_hx.T
    dh_prev = dz_t @ W_hh.T
    dW_hx = x_t.T @ dz_t
    dW_hh = h_prev.T @ dz_t
    db_h = dz_t.sum(axis=0)

    return dx_t, dh_prev, dW_hx, dW_hh, db_h

rng = np.random.default_rng(42)
batch_size, input_size, hidden_size = 2, 3, 4

x_t = rng.normal(size=(batch_size, input_size))
h_prev = rng.normal(size=(batch_size, hidden_size))
W_hx = rng.normal(scale=0.2, size=(input_size, hidden_size))
W_hh = rng.normal(scale=0.2, size=(hidden_size, hidden_size))
b_h = rng.normal(scale=0.1, size=(hidden_size,))
dh_next = rng.normal(size=(batch_size, hidden_size))

h_t, cache = rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h)
dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_cell_backward(dh_next, cache)

print("当前隐藏状态 h_t 的形状：", h_t.shape)
print("输入梯度 dx_t 的形状：", dx_t.shape)
print("上一隐藏状态梯度 dh_prev 的形状：", dh_prev.shape)
print("权重梯度 dW_hx 的形状：", dW_hx.shape)
print("权重梯度 dW_hh 的形状：", dW_hh.shape)
print("偏置梯度 db_h 的形状：", db_h.shape)
print("\nh_t =")
print(h_t)
print("\ndb_h =")
print(db_h)

# 用一个局部线性损失检查反向传播是否正确。
def local_loss(W_hx_candidate, W_hh_candidate, b_h_candidate):
    candidate_h, _ = rnn_cell_forward(x_t, h_prev, W_hx_candidate, W_hh_candidate, b_h_candidate)
    return float(np.sum(candidate_h * dh_next))

small_step = 1e-5
W_hx_plus = W_hx.copy()
W_hx_minus = W_hx.copy()
W_hx_plus[0, 0] += small_step
W_hx_minus[0, 0] -= small_step
numeric_dW_hx_00 = (local_loss(W_hx_plus, W_hh, b_h) - local_loss(W_hx_minus, W_hh, b_h)) / (2 * small_step)

W_hh_plus = W_hh.copy()
W_hh_minus = W_hh.copy()
W_hh_plus[0, 0] += small_step
W_hh_minus[0, 0] -= small_step
numeric_dW_hh_00 = (local_loss(W_hx, W_hh_plus, b_h) - local_loss(W_hx, W_hh_minus, b_h)) / (2 * small_step)

print("\n梯度检查 dW_hx[0, 0]：解析值 =", dW_hx[0, 0], "，数值值 =", numeric_dW_hx_00)
print("梯度检查 dW_hh[0, 0]：解析值 =", dW_hh[0, 0], "，数值值 =", numeric_dW_hh_00)


当前隐藏状态 h_t 的形状： (2, 4)
输入梯度 dx_t 的形状： (2, 3)
上一隐藏状态梯度 dh_prev 的形状： (2, 4)
权重梯度 dW_hx 的形状： (3, 4)
权重梯度 dW_hh 的形状： (4, 4)
偏置梯度 db_h 的形状： (4,)

h_t =
[[-0.0205 -0.1211 -0.075   0.0248]
 [-0.372   0.0837  0.4     0.2538]]

db_h =
[ 1.1202  0.8472 -0.5489 -0.2315]

梯度检查 dW_hx[0, 0]：解析值 = 0.49973043898522707 ，数值值 = 0.49973043898532404
梯度检查 dW_hh[0, 0]：解析值 = 0.330422803068365 ，数值值 = 0.3304228030687195


## 4 高级循环神经网络

### 4.1 理论计算题

设深度双向 RNN 有 $L$ 层，每层每个方向的隐藏单元数为 $H$，输入维度为 $D$，最终输出维度为 $O$。标准 RNN 单元每个方向每层包含 $W_{xh}$、$W_{hh}$ 和一个偏置。

第 1 层每个方向的输入维度是 $D$，参数量为

$$
DH+H^2+H.
$$

双向共有两个方向，所以第 1 层参数量为

$$
2(DH+H^2+H).
$$

第 2 层到第 $L$ 层，上一层双向输出拼接后维度为 $2H$，每个方向的参数量为

$$
(2H)H+H^2+H=3H^2+H.
$$

所以每一层参数量为

$$
2(3H^2+H)=6H^2+2H.
$$

最后输出层从 $2H$ 到 $O$，参数量为

$$
2HO+O.
$$

因此总参数量为

$$
\boxed{2(DH+H^2+H)+(L-1)(6H^2+2H)+2HO+O}
$$

如果按 PyTorch 的 nn.RNN 计数，每个方向每层有 bias_ih 和 bias_hh 两个偏置，比上式多 $2LH$ 个参数。


In [5]:
def deep_bidirectional_rnn_parameter_count(num_layers, hidden_size, input_dim, output_dim, use_pytorch_bias=False):
    """计算深度双向 RNN 加最终输出层的参数总数。"""
    if num_layers <= 0:
        raise ValueError("层数必须是正整数。")

    first_layer_params = 2 * (input_dim * hidden_size + hidden_size * hidden_size + hidden_size)
    later_layer_params = (num_layers - 1) * (6 * hidden_size * hidden_size + 2 * hidden_size)
    output_layer_params = 2 * hidden_size * output_dim + output_dim
    total_params = first_layer_params + later_layer_params + output_layer_params

    # PyTorch 的 RNN 每个方向每层有两个偏置；标准公式只合并成一个偏置。
    if use_pytorch_bias:
        total_params += 2 * num_layers * hidden_size

    return total_params

example_params = deep_bidirectional_rnn_parameter_count(num_layers=3, hidden_size=8, input_dim=5, output_dim=2)
example_params_pytorch = deep_bidirectional_rnn_parameter_count(num_layers=3, hidden_size=8, input_dim=5, output_dim=2, use_pytorch_bias=True)

print("示例参数：L=3，H=8，D=5，O=2")
print("标准单偏置写法的参数总数：", example_params)
print("PyTorch 双偏置写法的参数总数：", example_params_pytorch)


示例参数：L=3，H=8，D=5，O=2
标准单偏置写法的参数总数： 1058
PyTorch 双偏置写法的参数总数： 1106


### 4.2 编程题：双向 RNN 编码器

下面用 torch.nn.RNN 实现一个双向 RNN 编码器。输入 X 的形状为 (seq_len, batch, input_dim)，返回每个时间步拼接后的前向和后向隐藏状态，以及最终时间步的拼接隐藏状态作为序列表示。


In [6]:
class BiRNNEncoder(nn.Module):
    """使用 PyTorch RNN 实现的双向序列编码器。"""

    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            nonlinearity="tanh",
            bidirectional=True,
            batch_first=False,
        )

    def forward(self, X):
        # outputs 已经在最后一维拼接了前向和后向隐藏状态。
        outputs, h_n = self.rnn(X)

        # h_n 的原始形状为 (num_layers * 2, batch, hidden_dim)。
        h_n = h_n.view(self.num_layers, 2, X.shape[1], self.hidden_dim)
        last_forward = h_n[-1, 0]
        last_backward = h_n[-1, 1]
        sequence_representation = torch.cat([last_forward, last_backward], dim=-1)
        return outputs, sequence_representation

seq_len, batch_size, input_dim, hidden_dim = 5, 3, 4, 6
encoder = BiRNNEncoder(input_dim=input_dim, hidden_dim=hidden_dim, num_layers=2)
X = torch.randn(seq_len, batch_size, input_dim)
outputs, sequence_representation = encoder(X)

print("输入 X 的形状：", tuple(X.shape))
print("每个时间步的双向隐藏状态形状：", tuple(outputs.shape))
print("最终序列表示形状：", tuple(sequence_representation.shape))
print("输出形状是否满足要求：", tuple(outputs.shape) == (seq_len, batch_size, 2 * hidden_dim))
print("序列表示形状是否满足要求：", tuple(sequence_representation.shape) == (batch_size, 2 * hidden_dim))


输入 X 的形状： (5, 3, 4)
每个时间步的双向隐藏状态形状： (5, 3, 12)
最终序列表示形状： (3, 12)
输出形状是否满足要求： True
序列表示形状是否满足要求： True


## 5 嵌入向量

### 5.1 理论计算题

在 Skip-gram 模型中，给定中心词 $w_c$ 和真实上下文词 $w_o$。设中心词输入向量为 $v_c$，真实上下文词输出向量为 $u_o$，第 $k$ 个负样本词的输出向量为 $u_{n_k}$。

负采样把原来的多分类 softmax 问题改写为若干个二分类问题：真实上下文词判为正样本，采样得到的噪声词判为负样本。

对数似然目标为

$$
\log \sigma(u_o^{\top}v_c)+\sum_{k=1}^{K}\log \sigma(-u_{n_k}^{\top}v_c),
$$

所以最小化的负对数似然损失为

$$
\boxed{
\mathcal{L}= -\log \sigma(u_o^{\top}v_c)-\sum_{k=1}^{K}\log \sigma(-u_{n_k}^{\top}v_c)
}
$$

负样本通常从噪声分布中采样，常见选择是

$$
P_n(w)=\frac{f(w)^{3/4}}{\sum_{w'} f(w')^{3/4}}.
$$


### 5.2 编程题：CBOW 的完整 softmax 前向传播与损失

CBOW 使用上下文词向量的平均值作为隐藏表示，再通过输出矩阵得到词表上的 logits，最后用完整 softmax 计算目标中心词的交叉熵损失。


In [7]:
def stable_softmax(logits, axis=-1):
    """计算数值稳定的 softmax。"""
    shifted_logits = logits - np.max(logits, axis=axis, keepdims=True)
    exp_values = np.exp(shifted_logits)
    return exp_values / exp_values.sum(axis=axis, keepdims=True)


def cbow_forward_loss(context_indices, center_indices, W, W_out):
    """计算 CBOW 模型的前向传播和平均交叉熵损失。"""
    context_indices = np.asarray(context_indices, dtype=np.int64)
    center_indices = np.asarray(center_indices, dtype=np.int64)

    # 取出上下文词向量，并对每个样本的上下文向量取平均。
    context_vectors = W[context_indices]
    hidden = context_vectors.mean(axis=1)

    # 输出层得到词表大小维度的 logits，再转成概率分布。
    logits = hidden @ W_out
    probabilities = stable_softmax(logits, axis=1)

    batch_indices = np.arange(center_indices.shape[0])
    losses = -np.log(probabilities[batch_indices, center_indices] + 1e-12)
    loss = losses.mean()

    return loss, probabilities, hidden

words = ["我", "喜欢", "深度", "学习", "模型"]
word_to_id = {word: index for index, word in enumerate(words)}
V, embedding_dim = len(words), 3
rng = np.random.default_rng(7)
W = rng.normal(scale=0.2, size=(V, embedding_dim))
W_out = rng.normal(scale=0.2, size=(embedding_dim, V))

context_indices = np.array([
    [word_to_id["我"], word_to_id["喜欢"], word_to_id["学习"], word_to_id["模型"]],
    [word_to_id["喜欢"], word_to_id["深度"], word_to_id["学习"], word_to_id["模型"]],
])
center_indices = np.array([word_to_id["深度"], word_to_id["我"]])

loss, probabilities, hidden = cbow_forward_loss(context_indices, center_indices, W, W_out)

print("词到编号：", word_to_id)
print("上下文索引形状：", context_indices.shape)
print("隐藏表示形状：", hidden.shape)
print("输出概率分布形状：", probabilities.shape)
print("CBOW 平均交叉熵损失：", loss)

for sample_index, probability in enumerate(probabilities, start=1):
    readable_probability = {word: float(probability[word_to_id[word]]) for word in words}
    print(f"样本 {sample_index} 的预测概率：", readable_probability)


词到编号： {'我': 0, '喜欢': 1, '深度': 2, '学习': 3, '模型': 4}
上下文索引形状： (2, 4)
隐藏表示形状： (2, 3)
输出概率分布形状： (2, 5)
CBOW 平均交叉熵损失： 1.6266759057142526
样本 1 的预测概率： {'我': 0.19630926953516073, '喜欢': 0.20450809938148454, '深度': 0.19948864137946085, '学习': 0.20078881970570167, '模型': 0.1989051699981922}
样本 2 的预测概率： {'我': 0.19371760364472848, '喜欢': 0.20601098489423755, '深度': 0.19805332739331613, '学习': 0.2021759800578228, '模型': 0.20004210400989492}


## 6 注意力机制

### 6.1 理论计算题

给定 $Q\in\mathbb{R}^{2\times 4}$、$K\in\mathbb{R}^{3\times 4}$、$V\in\mathbb{R}^{3\times 5}$。缩放点积注意力为

$$
S=\frac{QK^{\top}}{\sqrt{d_k}},\qquad d_k=4.
$$

因此 $S=QK^{\top}/2$，形状为 $2\times 3$。然后逐行做 softmax：

$$
A=\text{softmax}(S),\qquad A\in\mathbb{R}^{2\times 3}.
$$

最后输出

$$
\boxed{O=AV=\text{softmax}\left(\frac{QK^{\top}}{2}\right)V}
$$

输出矩阵形状为 $2\times 5$。题目没有给出具体元素，所以理论结果用符号表示；下面用固定数值演示完整过程。


In [8]:
Q = np.array([
    [1.0, 0.0, 1.0, 0.0],
    [0.0, 1.0, 0.0, 1.0],
])
K = np.array([
    [1.0, 0.0, 0.0, 1.0],
    [0.0, 1.0, 1.0, 0.0],
    [1.0, 1.0, 0.0, 0.0],
])
V = np.array([
    [1.0, 2.0, 3.0, 4.0, 5.0],
    [2.0, 0.0, 1.0, 0.0, 2.0],
    [0.0, 1.0, 0.0, 1.0, 0.0],
])

scores = Q @ K.T / math.sqrt(Q.shape[1])
attention_weights = stable_softmax(scores, axis=1)
attention_output = attention_weights @ V

print("得分矩阵 S = QK^T / 2：")
print(scores)
print("\n注意力权重 A = softmax(S)：")
print(attention_weights)
print("\n注意力输出 O = A V：")
print(attention_output)
print("\n输出矩阵形状：", attention_output.shape)


得分矩阵 S = QK^T / 2：
[[0.5 0.5 0.5]
 [0.5 0.5 0.5]]

注意力权重 A = softmax(S)：
[[0.3333 0.3333 0.3333]
 [0.3333 0.3333 0.3333]]

注意力输出 O = A V：
[[1.     1.     1.3333 1.6667 2.3333]
 [1.     1.     1.3333 1.6667 2.3333]]

输出矩阵形状： (2, 5)


### 6.2 编程题：手写多头注意力前向传播

实现多头注意力。设 num_heads=2、d_model=4，则每个头的维度为 $d_k=d_v=d_{model}/num\_heads=2$。输入 X 的形状为 (seq_len, batch, d_model)，输出形状与输入一致。


In [9]:
class ManualMultiHeadAttention(nn.Module):
    """手写多头注意力的前向传播。"""

    def __init__(self, d_model=4, num_heads=2):
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError("d_model 必须能被 num_heads 整除。")

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def split_heads(self, tensor):
        """把最后一维拆成多个注意力头。"""
        seq_len, batch_size, _ = tensor.shape
        tensor = tensor.reshape(seq_len, batch_size, self.num_heads, self.d_k)
        return tensor.permute(1, 2, 0, 3)

    def merge_heads(self, tensor):
        """把多个注意力头重新拼接回模型维度。"""
        batch_size, num_heads, seq_len, head_dim = tensor.shape
        tensor = tensor.permute(2, 0, 1, 3).contiguous()
        return tensor.reshape(seq_len, batch_size, num_heads * head_dim)

    def forward(self, X):
        # 线性投影得到 Q、K、V，再拆分成多个头。
        Q = self.split_heads(self.W_q(X))
        K = self.split_heads(self.W_k(X))
        V = self.split_heads(self.W_v(X))

        # 每个头内部独立计算缩放点积注意力。
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        attention_weights = torch.softmax(scores, dim=-1)
        head_outputs = torch.matmul(attention_weights, V)

        # 拼接所有头，并通过最终线性层。
        concat_output = self.merge_heads(head_outputs)
        output = self.W_o(concat_output)
        return output, attention_weights

torch.manual_seed(42)
seq_len, batch_size, d_model = 3, 2, 4
X = torch.randn(seq_len, batch_size, d_model)
multi_head_attention = ManualMultiHeadAttention(d_model=4, num_heads=2)
output, attention_weights = multi_head_attention(X)

print("输入 X 的形状：", tuple(X.shape))
print("注意力权重的形状：", tuple(attention_weights.shape))
print("输出的形状：", tuple(output.shape))
print("输出形状是否与输入一致：", tuple(output.shape) == tuple(X.shape))
print("第一个时间步、第一个样本的输出向量：")
print(output[0, 0].detach().numpy())


输入 X 的形状： (3, 2, 4)
注意力权重的形状： (2, 2, 3, 3)
输出的形状： (3, 2, 4)
输出形状是否与输入一致： True
第一个时间步、第一个样本的输出向量：


[ 0.1555 -0.4513 -0.7055 -0.3643]